# Merge and Join


```
# Note: You can use .join to perform joins between multiple dataframes if they have indexes
df_all = df1.join([df2, df3], how="inner")

# Inner Join
merged_df = left_df.merge(right_df, on=['col1','col2'],
                          validate=None, # Validate used to verify 1:1, 1:N relationship
                          suffixes=('_left','_right'), how='inner')
# Left Join
merged_df = left_df.merge(right_df, on=['col1','col2'],
                          suffixes=('_left','_right'), how='left')
# Left Join on nearest value
merged_df = pd.merge_asof(left_df, right_df, on=['date'],
    suffixes=('_left','_right'),
    direction='nearest')
# Right Join
merged_df = left_df.merge(right_df, on=['col1','col2'],
                          suffixes=('_left','_right'), how='right')
# Outer Join
merged_df = left_df.merge(right_df, on=['col1','col2'],
                          suffixes=('_left','_right'), how='outer')

# Alternative outer join
merged_df = pd.merge_ordered(left_df, right_df, on='date',
                 suffixes=('_left','_right'),
                 fill_method='ffill')
# Alternative approach
merged_df = left_df.merge(right_df, left_on='left_col', right_on='right_col',
                          left_index=True, right_index=True, # if they are index
                          suffixes=('_left','_right'), how='outer')
# Semi Join
inner_join_df = left_df.merge(right_df, on='id')
semi_join_df = left_df[left_df['id'].isin(inner_join_df['id'])]

# Anti Join
inner_join_df = left_df.merge(right_df, on='id')
anti_join_df = left_df[~left_df['id'].isin(inner_join_df['id'])]

# Note : There are other keyword arguments in merge method for more flexibility
merged_df = left_df.merge(right_df, how='outer', # This will be an outer join
                          left_index=True,  # The left_df's index is used for joining
                          right_on='id', # The right_df's normal column is used for joining
                          suffixes=('_left', '_right'), # Duplicate columns from both dataframes will be suffixed
                          indicator=True, # A new column will tell which dataframe the value comes from
                          validate='1:m', # Validate whether left to right table joining has one-to-many relationship
                          sort="id") # The final merged dataframe will be sorted by this column

# Joining on exact value of Date column
merged_df = pd.merge_ordered(left_df, right_df, on="date_col",
                suffixes=['_left', '_right'], fill_method='ffill')

# Joining on approximate value of Date column
merged_df = pd.merge_asof(left_df, right_df, on="date_col",
                suffixes=['_left', '_right'],  direction='nearest')
```


# Duplicate values in python


```
# Drop complete duplicates
df.drop_duplicates(inplace = True)

# No of duplicates of specified column combinations
df.duplicated(['col1`', 'col2']).sum()

# Column names to check for partial duplicates
column_names = ['A','B','C']
duplicates = df.duplicated(subset = column_names, keep = False)
# See partial duplicate values
df[duplicates]

# Combine result for partial duplicates
summaries = {'D': 'max', 'E': 'mean'}
df = df.groupby(by = column_names).agg(summaries).reset_index()

#################### Record linkage ##########################
##### Used for Getting rid of duplicates from 2 different dataframes #######
import recordlinkage

# Create indexing object
indexer = recordlinkage.Index()

# Generate pairs blocked on index common in 2 dataframes
indexer.block('col')
pairs = indexer.index(df1, df2)
# See pairs
print(pairs)

# Create a Compare object
compare_cl = recordlinkage.Compare()

# Find exact matches for pairs of col1 and col2
compare_cl.exact('df1_col1', 'df2_col1', label='col1')
compare_cl.exact('df1_col2', 'df2_col2', label='col2')

# Find close matches for pairs of surname and address_1 using string similarity
compare_cl.string('df1_col3', 'df2_col3', threshold=0.85, label='col3')
compare_cl.string('df1_col4', 'df2_col4', threshold=0.85, label='col4')

# Find matches
potential_matches = compare_cl.compute(pairs, df1, df2)
# See potential matches
print(potential_matches)
# Filter matches where more than 2 columns match
matches = potential_matches[potential_matches.sum(axis = 1) => 2]
print(matches)
# See index
matches.index
# Get index of duplicates in df2
duplicate_rows = matches.index.get_level_values(1)
# Finding duplicates in df2
df2_duplicates = df2[df2.index.isin(duplicate_rows)]
# Finding rows in df2 that are not duplicates
df2_unique = df2[~df2.index.isin(duplicate_rows)]
# Link the DataFrames!
full_df = df1.append(df2_unique)
```
